# ___Updating the mycorrhizal states___
--------------------

In [1]:
!python --version

Python 3.14.2


The system cannot find the path specified.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [7]:
# https://datadryad.org/dataset/doi:10.5061/dryad.n8bm9
# THE SHEET "Original states data" HAS THE RAW DATA SCRAPED FROM PUBLICATIONS WITHOUT ANY INTEFERENCE FROM THE AUTHORS!!!!
maherali_original = pd.read_excel(r"../../data/chapter2/Maherali.etal.AmNat.Data.xlsx", sheet_name="Original states data", skiprows=range(2),
                                  usecols=("Source", "Original name (Genus species)", "Raw state record from publication"))
maherali_original.rename(mapper={old: old.replace('(', '').replace(')', '').lower().replace(' ', '_') for old in maherali_original.columns}, axis=1, inplace=True) # column names have parentheses and spaces
# taxonomy columns in Maherali et. al. dataset has trailing spaces :(
maherali_original.loc[:, "original_name_genus_species"] = maherali_original.original_name_genus_species.str.strip()
maherali_original.loc[:, "raw_state_record_from_publication"] = maherali_original.raw_state_record_from_publication.str.strip()
maherali_original.drop_duplicates(subset=("original_name_genus_species", "raw_state_record_from_publication"), inplace=True)

# final_maherali = pd.read_excel(r"../../data/chapter2/Maherali.etal.AmNat.Data.xlsx", sheet_name="Final list matched with phylo", skiprows=range(2))
# final_maherali.rename(mapper={old: old.lower().replace(' ', '_') for old in final_maherali.columns}, axis=1, inplace=True)
# final_maherali.genus_species = final_maherali.genus_species.str.strip().str.replace('_', ' ') # the sheet "Final list matched with phylo" has genus and specific epithets concatenated by under scores!

# in TRY, mycorrhiza type is trait id 7
try_myco = pd.read_csv(r"../../data/chapter2/TRY/mycorrhizal_states.txt", delimiter='\t', low_memory=False, encoding="latin1", usecols=["Dataset", "SpeciesName", "AccSpeciesName", "OrigValueStr",
                                "TraitID"]).dropna(subset=["AccSpeciesName", "OrigValueStr", "TraitID"])
# unify the mycorrhizal state info
# 'ECTO', 'NM/AM', 'EC', 'EC/AM', 'AM', 'Ecto', 'Non',        'vesicular-arbuscular mycorrhiza', 'ectomycorrhiza', 'no', '0', 'Ph.th.end.', 'VAM', 'Ectomycorrhiza', 'E.ch.ect.', 'arbuscular',
# 'ec?', 'VA', 'ecto', 'Absent', 'non-ectomycorrhizal', 'ectomycorrhizal', 'Yes', 'No', 'EM', 'AMNM', 'NM', 'AM + EM', 'ERM', 'Ericoid', 'ECM'

MYCORRHIZAL_STATES_REPLACEMENTS = {
    "ECTO": "EcM",
    "Ecto": "EcM",
    "EC": "EcM",
    "ectomycorrhiza": "EcM",
    "Ectomycorrhiza": "EcM",
    "ecto": "EM",
    "ectomycorrhizal": "EcM",
    "ECM": "EcM",
    "vesicular-arbuscular mycorrhiza" : "AM",
    "VAM": "AM",
    "VA": "AM",
    "Non": "NM",
    "AMNM": "NM/AM",
    "Ericoid": "ErM",
    "ERM": "ErM",
    "AM + EM": "AM/EcM",
    "EC/AM": "AM/EcM",
    "Orchid": "OrM",
    "OrM": "OrM"
}

try_myco.loc[:, "OrigValueStr"] = try_myco.OrigValueStr.replace(MYCORRHIZAL_STATES_REPLACEMENTS)
try_myco = try_myco.query("OrigValueStr.isin(@MYCORRHIZAL_STATES_REPLACEMENTS.values())")

# scrape the online only MycoDB metadata and serialize it to the disk
# req = Request(url=r"https://www.nature.com/articles/sdata201628/tables/2", headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:142.0) Gecko/20100101 Firefox/142.0"})
# with urlopen(req) as r:
#     soup = BeautifulSoup(r.read())
# 
# table = soup.find(name="table", attrs={"class": "data last-table"}) # locate the metadata table
# [th.text.strip() for th in table.find_all(name="th")] # column names
# mycodb_descriptions = [[td.text for td in tr.find_all(name="td")] for tr in table.find_all(name="tr")[1:]] # parse the rows
# 
# # create a dataframe using the parsed rows and column names and serialize it to the disk
# pd.DataFrame({ 
#     "Variable": [row[0] for row in mycodb_descriptions],
#     "Description": [row[1] for row in mycodb_descriptions],
#     "Variable Type (range)": [row[2] for row in mycodb_descriptions],
#     "Levels (#studies/level)": [row[3] for row in mycodb_descriptions],
# }).to_csv(r"../data/chapter2/MycoDB_version4_metadata.csv", index=False)

# species that have info in FRED v3 for first order fine root traits RD and SRL
collab_categorical = pd.read_csv(r"../../data/chapter2/FREDv3subset/FRED_subset_collab_categorical.csv") 

# even though we had missing data for photosynthetic pathways and mycorrhizal states in the records that had data for the 4 chosen traits in FRED, FRED could still have info for the categorical traits in other records 
# that were filtered out due to not having all the 4 trait values?????
fred_myco = pd.read_csv(r"../../data/chapter2/FRED/FRED3_Entire_Database_2021.csv", low_memory=False, header=0, skiprows=range(1, 10), encoding="latin1",
                        usecols=("F01286", "F01287", "F00645", "F00004")).dropna(subset=("F01286", "F01287", "F00645")).drop_duplicates()
# concatenate the genus name and specific epithet to introduce a column for binominal name
fred_myco.insert(loc=0, column="binominal", value=fred_myco.F01286.str.strip().str.capitalize() + ' ' + fred_myco.F01287.str.strip().str.lower())

In [12]:
species_of_interest = collab_categorical.loc[:, ["F01286", "F01287"]].drop_duplicates().agg(' '.join, axis=1).str.strip().reset_index(drop=True)
species_of_interest

0              Acer saccharum
1          Fraxinus americana
2             Viola pubescens
3      Hydrophyllum canadense
4              Larix gmelinii
                ...          
390            Acer triflorum
391          Pinus massoniana
392           Malus domestica
393            Prunus persica
394            Vitis vinifera
Length: 395, dtype: object

### ___Maherali, H. et al. (2016)___
------------------------------

In [19]:
maherali_original_ = maherali_original.query("original_name_genus_species.isin(@species_of_interest)").drop_duplicates(subset=["original_name_genus_species", "raw_state_record_from_publication"])
maherali_original_

,source,original_name_genus_species,raw_state_record_from_publication
29,Akhmetzhanova et al. 2012,Abies nephrolepis,EM
70,Wang&Qiu2006,Acacia auriculiformis,AM
80,Wang&Qiu2006,Acacia mangium,EM AM
99,Akhmetzhanova et al. 2012,Acer barbinerve,EM
108,Akhmetzhanova et al. 2012,Acer davidii,AM
...,...,...,...
11958,Akhmetzhanova et al. 2012,Ulmus pumila,NM
11985,Wang&Qiu2006,Vaccinium corymbosum,ERM
12240,Akhmetzhanova et al. 2012,Veronica spuria,AM
12444,Wang&Qiu2006,Vitis vinifera,AM


### ___TRY___
--------------------

In [21]:
try_myco_ = try_myco.query("AccSpeciesName.isin(@species_of_interest)").drop_duplicates(subset=["AccSpeciesName", "OrigValueStr"])
try_myco_

,Dataset,SpeciesName,AccSpeciesName,TraitID,OrigValueStr
7,Abisko & Sheffield Database,Caltha palustris,Caltha palustris,7.0,NM/AM
18,Abisko & Sheffield Database,Pinus sylvestris,Pinus sylvestris,7.0,EcM
20,Abisko & Sheffield Database,Populus tremula,Populus tremula,7.0,EcM
33,Abisko & Sheffield Database,Sorbus aucuparia,Sorbus aucuparia,7.0,NM/AM
49,Sheffield Database,Quercus robur,Quercus robur,7.0,EcM
...,...,...,...,...,...
1143409,Independent evolutionary changes in fine-root ...,Castanea henryi,Castanea henryi,7.0,EcM
1143433,Independent evolutionary changes in fine-root ...,Castanopsis wattii,Castanopsis wattii,7.0,EcM
1143481,Independent evolutionary changes in fine-root ...,Lithocarpus chiungchungensis,Lithocarpus chiungchungensis,7.0,EcM
1143583,Independent evolutionary changes in fine-root ...,Quercus serrata,Quercus serrata,7.0,EcM


### ___FRED___
-------------------------------

In [29]:
# again, since this is FRED, look only for species that we do not have mycorrhizal state info for in the subset
missing_states_collab_species = collab_categorical.query("F00645.isna()").loc[:, ["F01286", "F01287"]].drop_duplicates().agg(' '.join, axis=1).str.strip().reset_index(drop=True)
missing_states_collab_species

0             Acer saccharum
1         Fraxinus americana
2            Viola pubescens
3     Hydrophyllum canadense
4             Larix gmelinii
               ...          
91           Ulmus americana
92        Betula platyphylla
93           Malus domestica
94            Prunus persica
95            Vitis vinifera
Length: 96, dtype: object

In [35]:
fred_myco_ = fred_myco.query("binominal.isin(@missing_states_collab_species) and (F00645!='mycorrhizal')").drop_duplicates(subset=["binominal", "F00645"])
fred_myco_

,binominal,F00004,F01286,F01287,F00645
0,Dicranopteris linearis,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",Dicranopteris,linearis,AM
3,Cunninghamia lanceolata,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",Cunninghamia,lanceolata,AM
6,Magnolia baillonii,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",Magnolia,baillonii,AM
11,Acacia auriculiformis,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",Acacia,auriculiformis,AM
15,Gordonia axillaris,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",Gordonia,axillaris,AM
...,...,...,...,...,...
53506,Vitis vinifera,"Akhmetzhanova AA, Soudzilovskaiana NA, Onipche...",Vitis,vinifera,AM
54225,Acer pictum,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",Acer,pictum,AM
54266,Castanopsis faberi,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",Castanopsis,faberi,EM
54281,Elaeocarpus sylvestris,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",Elaeocarpus,sylvestris,AM


### ___GRoot___
----------------------------

In [42]:
pd.read_csv(r"../../data/chapter2/GRooTFullVersion.csv", low_memory=False, encoding="latin1")

,GRooTID,source,versionSource,originalID,referencesAbbreviated,references,referencesDataset,referencesAdditional,family,genus,...,belowgroundEntitiesOrder,belowgroundEntitiesOrderMin,belowgroundEntitiesOrderMax,belowgroundEntitiesFunctional,belowgroundEntitiesDiameterMin,belowgroundEntitiesDiameterMax,traitName,traitValue,errorRiskEntries,errorRisk
0,1,FRED,2.0,33116.0,Aaltonen 1920,Aaltonen VT. 1920. Abreiten Der Forstwissensch...,"Fan Y, Miguez-Macho G, Jobbagy EG, Jackson RB,...",NaN,NaN,Betula,...,NaN,NaN,NaN,NaN,NaN,NaN,Rooting_depth,0.850,NaN,NaN
1,2,FRED,2.0,33115.0,Aaltonen 1920,Aaltonen VT. 1920. Abreiten Der Forstwissensch...,"Fan Y, Miguez-Macho G, Jobbagy EG, Jackson RB,...",NaN,NaN,Picea,...,NaN,NaN,NaN,NaN,NaN,NaN,Rooting_depth,0.900,NaN,NaN
2,3,FRED,2.0,33113.0,Aaltonen 1920,Aaltonen VT. 1920. Abreiten Der Forstwissensch...,"Fan Y, Miguez-Macho G, Jobbagy EG, Jackson RB,...",NaN,NaN,Pinus,...,NaN,NaN,NaN,NaN,NaN,NaN,Rooting_depth,1.000,NaN,NaN
3,4,FRED,2.0,33112.0,Aaltonen 1920,Aaltonen VT. 1920. Abreiten Der Forstwissensch...,"Fan Y, Miguez-Macho G, Jobbagy EG, Jackson RB,...",NaN,NaN,Pinus,...,NaN,NaN,NaN,NaN,NaN,NaN,Rooting_depth,0.980,NaN,NaN
4,5,FRED,2.0,33114.0,Aaltonen 1920,Aaltonen VT. 1920. Abreiten Der Forstwissensch...,"Fan Y, Miguez-Macho G, Jobbagy EG, Jackson RB,...",NaN,NaN,Pinus,...,NaN,NaN,NaN,NaN,NaN,NaN,Rooting_depth,0.560,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114217,50979,Roumet,NaN,NaN,Roumet_unpublished,Roumet_unpublished,NaN,NaN,Asteraceae,Inula,...,NaN,NaN,NaN,NaN,NaN,NaN,Root_C_concentration,389.000,2.0,0.397859
114218,50979,Roumet,NaN,NaN,Roumet_unpublished,Roumet_unpublished,NaN,NaN,Asteraceae,Inula,...,NaN,NaN,NaN,NaN,NaN,NaN,Root_dry_matter_content,0.106,2.0,0.978497
114219,50979,Roumet,NaN,NaN,Roumet_unpublished,Roumet_unpublished,NaN,NaN,Asteraceae,Inula,...,NaN,NaN,NaN,NaN,NaN,NaN,Root_mass_fraction,0.453,1.0,0.000000
114220,50979,Roumet,NaN,NaN,Roumet_unpublished,Roumet_unpublished,NaN,NaN,Asteraceae,Inula,...,NaN,NaN,NaN,NaN,NaN,NaN,Root_N_concentration,27.800,3.0,-1.643390


In [36]:
#-----------------------------------------------------------
# NOW COMBINE ALL OF THESE TO CREATE A SINGLE DATASET
#-----------------------------------------------------------

In [40]:
data = pd.merge(left=maherali_original_.rename({"original_name_genus_species": "binominal"}, axis=1), left_on="binominal",
         right=try_myco_.rename({"AccSpeciesName": "binominal"}, axis=1), right_on="binominal", how="outer").drop(["SpeciesName", "TraitID"], axis=1)
data

,source,binominal,raw_state_record_from_publication,Dataset,OrigValueStr
0,Akhmetzhanova et al. 2012,Abies nephrolepis,EM,Mycorrhizal Intensity Database Across the Form...,EcM
1,Akhmetzhanova et al. 2012,Abies nephrolepis,EM,FRED - Fine Root Ecology Database,EM
2,Wang&Qiu2006,Acacia auriculiformis,AM,Global 15N Database,AM
3,Wang&Qiu2006,Acacia auriculiformis,AM,Mycorrhiza Database,EcM
4,NaN,Acacia crassicarpa,NaN,FRED - Fine Root Ecology Database,AM/EcM
...,...,...,...,...,...
544,Wang&Qiu2006,Vaccinium corymbosum,ERM,Independent evolutionary changes in fine-root ...,ErM
545,NaN,Vaccinium mandarinorum,NaN,FRED - Fine Root Ecology Database,ErM
546,Akhmetzhanova et al. 2012,Veronica spuria,AM,Mycorrhizal Intensity Database Across the Form...,AM
547,Wang&Qiu2006,Vitis vinifera,AM,FRED - Fine Root Ecology Database,AM
